In [1]:
import numpy as np
import sionna
from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver
import sionna.rt.scene as scenes
import json

with open('config.json') as f:
    cfg = json.load(f)

scene_ref = cfg['scene']
if '/' in scene_ref or scene_ref.endswith('.xml'):
    scene = load_scene(scene_ref)
else:
    scene = load_scene(getattr(scenes, scene_ref))

scene.frequency = cfg['frequency']
scene.synthetic_array = True

scene.tx_array = PlanarArray(
    num_rows=cfg['tx_array']['rows'],
    num_cols=cfg['tx_array']['cols'],
    vertical_spacing=cfg['tx_array']['spacing'],
    horizontal_spacing=cfg['tx_array']['spacing'],
    pattern='iso', polarization='V'
)
scene.rx_array = PlanarArray(
    num_rows=cfg['rx_array']['rows'],
    num_cols=cfg['rx_array']['cols'],
    vertical_spacing=cfg['rx_array']['spacing'],
    horizontal_spacing=cfg['rx_array']['spacing'],
    pattern='iso', polarization='V'
)

for i, pos in enumerate(cfg['transmitters']):
    tx = Transmitter(name=f'gnb-{i}', position=pos)
    scene.add(tx)

for i, pos in enumerate(cfg['static_receivers']):
    rx = Receiver(name=f'ue-{i}', position=pos)
    scene.add(rx)

print(f'Scene loaded: {scene_ref}')
print(f'gNBs: {len(cfg["transmitters"])}  |  Static UEs: {len(cfg["static_receivers"])}')

Scene loaded: car_factory_scene/car_factory.xml
gNBs: 10  |  Static UEs: 40


In [2]:
# Interactive 3D WebGL preview — rotate/zoom with mouse
scene.preview()

In [2]:
# Compute ray paths and overlay them on the preview
from sionna.rt import PathSolver
path_solver = PathSolver()
paths = path_solver(scene, max_depth=3)
scene.preview(paths=paths)